# Copernicus Solar Forecasting — notebook 2 : métriques, baselines et premiers modèles tabulaires

Ce notebook poursuit le travail du notebook 1 en se concentrant sur :

- la mise en place de l'évaluation ;
- la construction de baselines simples mais pertinentes ;
- l'entraînement des premiers modèles supervisés classiques sur les features tabulaires ;
- la comparaison des performances sur un split chronologique train / validation.

## Objectifs
- vérifier que les métriques sont bien définies et exploitables ;
- établir une ou plusieurs **baselines de référence** ;
- tester plusieurs modèles classiques :
  - Ridge,
  - Elastic Net,
  - Random Forest,
  - HistGradientBoosting ;
- comparer les performances globales et par horizon ;
- identifier une première famille de modèles prometteuse pour la suite du projet.

## Imports et configurations

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)
plt.style.use("default")
warnings.filterwarnings("ignore")

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
CANDIDATE_ROOTS = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/mnt/data"),
]

PROJECT_ROOT = None
for candidate in CANDIDATE_ROOTS:
    if (candidate / "config.py").exists() or (candidate / "src").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = c:\Users\alixg\OneDrive - Université Paris-Dauphine\M2 Quant\Machine Learning\projet\copernicus_solar_forecasting


In [4]:
from config import (
    FORECAST_HORIZONS_MINUTES,
    VALIDATION_FRACTION,
    INPUT_VARIABLES,
)

from src.data_loading import open_processed_profile, extract_roi
from src.features import build_physical_inputs, build_tabular_features, build_advanced_features
from src.preprocessing import (
    temporal_train_validation_split,
    fit_standardizer,
    transform_with_standardizer,
)

from src.metrics import (
    evaluate_forecasts,
    evaluate_by_horizon,
    evaluate_spatial_means,
    evaluate_model_bundle,
    compare_global_scores,
    add_skill_scores,
)

from src.baselines import (
    persistence_last_ghi_baseline,
    persistence_csi_baseline,
    mean_image_baseline,
)

from models.models_tabular import (
    flatten_target,
    prepare_tabular_inputs,
    fit_ridge_multioutput,
    fit_elasticnet_multioutput,
    fit_random_forest_multioutput,
    fit_hist_gb_multioutput,
    predict_tensor,
)

from src.visualization import plot_error_analysis

## Paramètres du notebook

In [5]:
# %% Paramètres

PROFILE_FOR_MODELING = "dev"   # mettre "sample" ou "full" plus tard
RANDOM_STATE = 42

RIDGE_ALPHA = 1.0

ELASTICNET_ALPHA = 0.001
ELASTICNET_L1_RATIO = 0.5

RF_N_ESTIMATORS = 50
RF_MAX_DEPTH = 10

HGB_MAX_DEPTH = 6
HGB_LEARNING_RATE = 0.05
HGB_MAX_ITER = 200

## 1. Chargement des données et reconstruction du pipeline

On recharge le profil processed principal puis on reconstruit les objets nécessaires :
- split train / validation chronologique ;
- représentation physique ;
- standardisation ajustée sur le train uniquement ;
- features tabulaires ;
- cibles train / validation.

In [6]:
# %% Chargement du profil train

data = open_processed_profile(PROFILE_FOR_MODELING, split="train")
print("Variables disponibles :", list(data["X"].keys()))
print("Shape target :", data["y"].shape)

Variables disponibles : ['GHI', 'CLS', 'SZA', 'SAA']
Shape target : (32, 4, 51, 51)


In [7]:
# %% Split chronologique train / validation

train_idx, val_idx = temporal_train_validation_split(
    n_samples=len(data["indices"]),
    validation_fraction=VALIDATION_FRACTION,
)

print("n_train =", len(train_idx))
print("n_val   =", len(val_idx))

n_train = 25
n_val   = 7


In [8]:
# %% Jeux bruts train / val

train_arrays_raw = {name: values[train_idx] for name, values in data["X"].items()}
val_arrays_raw = {name: values[val_idx] for name, values in data["X"].items()}

train_target = data["y"][train_idx]
val_target = data["y"][val_idx]

print("Train target :", train_target.shape)
print("Val target   :", val_target.shape)

Train target : (25, 4, 51, 51)
Val target   : (7, 4, 51, 51)


In [9]:
# %% Représentation physique

train_arrays_phys = build_physical_inputs(
    train_arrays_raw,
    keep_raw_ghi=False,
    encode_angles=True,
)

val_arrays_phys = build_physical_inputs(
    val_arrays_raw,
    keep_raw_ghi=False,
    encode_angles=True,
)

print("Variables physiques :", list(train_arrays_phys.keys()))

Variables physiques : ['CSI', 'CLS', 'SZA_sin', 'SZA_cos', 'SAA_sin', 'SAA_cos']


In [10]:
X_train_adv = build_advanced_features(train_arrays_raw)
X_val_adv = build_advanced_features(val_arrays_raw)

print(f"Nouvelle forme des features : {X_train_adv.shape}")

Nouvelle forme des features : (25, 16)


In [11]:
# %% Standardisation fit sur train seulement

standardizer = fit_standardizer(
    train_arrays_phys,
    variables=train_arrays_phys.keys(),
)

train_arrays_std = transform_with_standardizer(train_arrays_phys, standardizer)
val_arrays_std = transform_with_standardizer(val_arrays_phys, standardizer)

In [12]:
# %% Features tabulaires

tabular_features_train = build_tabular_features(train_arrays_std)
tabular_features_val = build_tabular_features(val_arrays_std)

print("Tabular train shape :", tabular_features_train.shape)
print("Tabular val shape   :", tabular_features_val.shape)

display(tabular_features_train.head())

Tabular train shape : (25, 502)
Tabular val shape   : (7, 502)


,CSI_mean_t0,CSI_std_t0,CSI_min_t0,CSI_max_t0,CSI_mean_t1,CSI_std_t1,CSI_min_t1,CSI_max_t1,CSI_mean_t2,CSI_std_t2,CSI_min_t2,CSI_max_t2,CSI_mean_t3,CSI_std_t3,CSI_min_t3,CSI_max_t3,CSI_mean_global,CSI_std_global,CSI_trend_last_minus_first,CSI_q25_t0,CSI_q50_t0,CSI_q75_t0,CSI_q25_t1,CSI_q50_t1,CSI_q75_t1,CSI_q25_t2,CSI_q50_t2,CSI_q75_t2,CSI_q25_t3,CSI_q50_t3,CSI_q75_t3,CSI_q1_mean_t0,CSI_q1_mean_t1,CSI_q1_mean_t2,CSI_q1_mean_t3,CSI_q2_mean_t0,CSI_q2_mean_t1,CSI_q2_mean_t2,CSI_q2_mean_t3,CSI_q3_mean_t0,CSI_q3_mean_t1,CSI_q3_mean_t2,CSI_q3_mean_t3,CSI_q4_mean_t0,CSI_q4_mean_t1,CSI_q4_mean_t2,CSI_q4_mean_t3,CLS_mean_t0,CLS_std_t0,CLS_min_t0,CLS_max_t0,CLS_mean_t1,CLS_std_t1,CLS_min_t1,CLS_max_t1,CLS_mean_t2,CLS_std_t2,CLS_min_t2,CLS_max_t2,CLS_mean_t3,CLS_std_t3,CLS_min_t3,CLS_max_t3,CLS_mean_t4,CLS_std_t4,CLS_min_t4,CLS_max_t4,CLS_mean_t5,CLS_std_t5,CLS_min_t5,CLS_max_t5,CLS_mean_t6,CLS_std_t6,CLS_min_t6,CLS_max_t6,CLS_mean_t7,CLS_std_t7,CLS_min_t7,CLS_max_t7,CLS_mean_global,CLS_std_global,CLS_trend_last_minus_first,CLS_q25_t0,CLS_q50_t0,CLS_q75_t0,CLS_q25_t1,CLS_q50_t1,CLS_q75_t1,CLS_q25_t2,CLS_q50_t2,CLS_q75_t2,CLS_q25_t3,CLS_q50_t3,CLS_q75_t3,CLS_q25_t4,CLS_q50_t4,CLS_q75_t4,CLS_q25_t5,CLS_q50_t5,CLS_q75_t5,...,SAA_sin_q3_mean_t7,SAA_sin_q4_mean_t0,SAA_sin_q4_mean_t1,SAA_sin_q4_mean_t2,SAA_sin_q4_mean_t3,SAA_sin_q4_mean_t4,SAA_sin_q4_mean_t5,SAA_sin_q4_mean_t6,SAA_sin_q4_mean_t7,SAA_cos_mean_t0,SAA_cos_std_t0,SAA_cos_min_t0,SAA_cos_max_t0,SAA_cos_mean_t1,SAA_cos_std_t1,SAA_cos_min_t1,SAA_cos_max_t1,SAA_cos_mean_t2,SAA_cos_std_t2,SAA_cos_min_t2,SAA_cos_max_t2,SAA_cos_mean_t3,SAA_cos_std_t3,SAA_cos_min_t3,SAA_cos_max_t3,SAA_cos_mean_t4,SAA_cos_std_t4,SAA_cos_min_t4,SAA_cos_max_t4,SAA_cos_mean_t5,SAA_cos_std_t5,SAA_cos_min_t5,SAA_cos_max_t5,SAA_cos_mean_t6,SAA_cos_std_t6,SAA_cos_min_t6,SAA_cos_max_t6,SAA_cos_mean_t7,SAA_cos_std_t7,SAA_cos_min_t7,SAA_cos_max_t7,SAA_cos_mean_global,SAA_cos_std_global,SAA_cos_trend_last_minus_first,SAA_cos_q25_t0,SAA_cos_q50_t0,SAA_cos_q75_t0,SAA_cos_q25_t1,SAA_cos_q50_t1,SAA_cos_q75_t1,SAA_cos_q25_t2,SAA_cos_q50_t2,SAA_cos_q75_t2,SAA_cos_q25_t3,SAA_cos_q50_t3,SAA_cos_q75_t3,SAA_cos_q25_t4,SAA_cos_q50_t4,SAA_cos_q75_t4,SAA_cos_q25_t5,SAA_cos_q50_t5,SAA_cos_q75_t5,SAA_cos_q25_t6,SAA_cos_q50_t6,SAA_cos_q75_t6,SAA_cos_q25_t7,SAA_cos_q50_t7,SAA_cos_q75_t7,SAA_cos_q1_mean_t0,SAA_cos_q1_mean_t1,SAA_cos_q1_mean_t2,SAA_cos_q1_mean_t3,SAA_cos_q1_mean_t4,SAA_cos_q1_mean_t5,SAA_cos_q1_mean_t6,SAA_cos_q1_mean_t7,SAA_cos_q2_mean_t0,SAA_cos_q2_mean_t1,SAA_cos_q2_mean_t2,SAA_cos_q2_mean_t3,SAA_cos_q2_mean_t4,SAA_cos_q2_mean_t5,SAA_cos_q2_mean_t6,SAA_cos_q2_mean_t7,SAA_cos_q3_mean_t0,SAA_cos_q3_mean_t1,SAA_cos_q3_mean_t2,SAA_cos_q3_mean_t3,SAA_cos_q3_mean_t4,SAA_cos_q3_mean_t5,SAA_cos_q3_mean_t6,SAA_cos_q3_mean_t7,SAA_cos_q4_mean_t0,SAA_cos_q4_mean_t1,SAA_cos_q4_mean_t2,SAA_cos_q4_mean_t3,SAA_cos_q4_mean_t4,SAA_cos_q4_mean_t5,SAA_cos_q4_mean_t6,SAA_cos_q4_mean_t7
0,-0.147072,0.246611,-1.681128,-0.010976,0.001575,0.264283,-2.080225,0.127459,0.103543,0.309742,-2.280185,0.233526,0.199936,0.294195,-2.119135,0.318208,0.039496,0.307913,0.347008,-0.095457,-0.073610,-0.055990,0.063944,0.078670,0.091392,0.180850,0.191738,0.202580,0.270456,0.279876,0.290221,-0.297348,-0.157087,-0.113018,-0.017534,-0.132270,0.025628,0.161101,0.256827,-0.119171,0.033429,0.147855,0.253059,-0.049193,0.094509,0.205815,0.295218,-1.891109,0.039605,-2.025005,-1.761102,-1.570820,0.042075,-1.710103,-1.412795,-1.255072,0.044353,-1.396612,-1.072579,-0.949160,0.046547,-1.091063,-0.745429,-0.656869,0.048738,-0.797764,-0.434164,-0.381215,0.050979,-0.520128,-0.141310,-0.124632,0.053284,-0.260679,0.132763,0.110851,0.055644,-0.021719,0.385159,-0.839753,0.661570,2.001959,-1.910684,-1.886632,-1.867925,-1.591477,-1.568910,-1.548792,-1.277169,-1.254973,-1.234113,-0.973179,-0.951132,-0.930049,-0.683295,-0.660950,-0.639274,-0.409964,-0.387397,-0.364904,...,-0.033545,1.564817,1.384645,1.175580,0.943301,0.693819,0.433516,0.168937,-0.093210,-1.914863,0.061407,-2.06136

In [13]:
# %% Conversion vers numpy pour modèles tabulaires

X_train_tab, X_val_tab, feature_columns = prepare_tabular_inputs(
    tabular_features_train,
    tabular_features_val,
)

y_train_flat = flatten_target(train_target)

print("X_train_tab :", X_train_tab.shape)
print("X_val_tab   :", X_val_tab.shape)
print("y_train_flat:", y_train_flat.shape)
print("Nombre de features tabulaires :", len(feature_columns))

X_train_tab : (25, 502)
X_val_tab   : (7, 502)
y_train_flat: (25, 10404)
Nombre de features tabulaires : 502


## 2. Métriques d'évaluation

Les modèles seront comparés avec :
- **RMSE** global,
- **MAE** global,
- **RMSE / MAE par horizon**,
- et **scores sur moyennes spatiales**.

Cette combinaison permet de distinguer :
- l'erreur globale sur le tenseur,
- la difficulté croissante selon l'horizon,
- et la capacité à prédire le niveau moyen d'irradiance indépendamment des détails spatiaux fins.

In [14]:
# %% Vérification rapide des métriques sur une prédiction nulle

dummy_pred = np.zeros_like(val_target)

print("Scores globaux - prédiction nulle")
display(evaluate_forecasts(val_target, dummy_pred))

print("Scores par horizon - prédiction nulle")
display(evaluate_by_horizon(val_target, dummy_pred))

print("Scores sur moyenne spatiale - prédiction nulle")
display(evaluate_spatial_means(val_target, dummy_pred))

Scores globaux - prédiction nulle


,metric,value
0,RMSE,392.485415
1,MAE,367.200385
2,R2,-7.019542
3,MAPE,100.000000


Scores par horizon - prédiction nulle


,horizon_min,RMSE,MAE
0,15,398.994495,380.848920
1,30,395.346910,373.443357
2,45,390.693096,363.697722
3,60,384.762683,350.811542


Scores sur moyenne spatiale - prédiction nulle


,horizon_min,RMSE_spatial_mean,MAE_spatial_mean
0,15,397.920569,380.848920
1,30,394.299381,373.443357
2,45,389.692109,363.697722
3,60,383.795096,350.811542


## 3. Baseline 1 — persistance brute

Première baseline : on suppose que les futures images ressemblent à la dernière image observée de `GHI`.

Formellement :

$$
\hat{Y}_{t+h} = GHI_t
$$

pour chacun des horizons futurs.

In [15]:
# %% Baseline persistance brute

y_pred_persistence_raw = persistence_last_ghi_baseline(val_arrays_raw)

print("Shape y_pred_persistence_raw :", y_pred_persistence_raw.shape)

baseline_raw_results = evaluate_model_bundle(
    val_target,
    y_pred_persistence_raw,
    model_name="persistence_raw",
)

print("Scores globaux")
display(baseline_raw_results["global"])

print("Scores par horizon")
display(baseline_raw_results["by_horizon"])

print("Scores sur moyenne spatiale")
display(baseline_raw_results["spatial_means"])

Shape y_pred_persistence_raw : (7, 4, 51, 51)
Scores globaux


,metric,value,model
0,RMSE,77.652985,persistence_raw
1,MAE,63.439056,persistence_raw
2,R2,0.686080,persistence_raw
3,MAPE,27.925336,persistence_raw


Scores par horizon


,horizon_min,RMSE,MAE,model
0,15,28.998959,25.867481,persistence_raw
1,30,57.192327,51.134476,persistence_raw
2,45,84.976879,75.965643,persistence_raw
3,60,113.079497,100.788624,persistence_raw


Scores sur moyenne spatiale


,horizon_min,RMSE_spatial_mean,MAE_spatial_mean,model
0,15,27.887018,25.339949,persistence_raw
1,30,55.861563,50.583423,persistence_raw
2,45,83.284027,75.247970,persistence_raw
3,60,111.344196,99.886624,persistence_raw


In [16]:
# %% Baseline persistance brute

y_pred_persistence_raw = persistence_last_ghi_baseline(val_arrays_raw)

print("Shape y_pred_persistence_raw :", y_pred_persistence_raw.shape)

baseline_raw_results = evaluate_model_bundle(
    val_target,
    y_pred_persistence_raw,
    model_name="persistence_raw",
)

print("Scores globaux")
display(baseline_raw_results["global"])

print("Scores par horizon")
display(baseline_raw_results["by_horizon"])

print("Scores sur moyenne spatiale")
display(baseline_raw_results["spatial_means"])

Shape y_pred_persistence_raw : (7, 4, 51, 51)
Scores globaux


,metric,value,model
0,RMSE,77.652985,persistence_raw
1,MAE,63.439056,persistence_raw
2,R2,0.686080,persistence_raw
3,MAPE,27.925336,persistence_raw


Scores par horizon


,horizon_min,RMSE,MAE,model
0,15,28.998959,25.867481,persistence_raw
1,30,57.192327,51.134476,persistence_raw
2,45,84.976879,75.965643,persistence_raw
3,60,113.079497,100.788624,persistence_raw


Scores sur moyenne spatiale


,horizon_min,RMSE_spatial_mean,MAE_spatial_mean,model
0,15,27.887018,25.339949,persistence_raw
1,30,55.861563,50.583423,persistence_raw
2,45,83.284027,75.247970,persistence_raw
3,60,111.344196,99.886624,persistence_raw


In [18]:
baseline_raw_results

{'global':   metric      value            model
 0   RMSE  77.652985  persistence_raw
 1    MAE  63.439056  persistence_raw
 2     R2   0.686080  persistence_raw
 3   MAPE  27.925336  persistence_raw,
 'by_horizon':    horizon_min        RMSE         MAE            model
 0           15   28.998959   25.867481  persistence_raw
 1           30   57.192327   51.134476  persistence_raw
 2           45   84.976879   75.965643  persistence_raw
 3           60  113.079497  100.788624  persistence_raw,
 'spatial_means':    horizon_min  RMSE_spatial_mean  MAE_spatial_mean            model
 0           15          27.887018         25.339949  persistence_raw
 1           30          55.861563         50.583423  persistence_raw
 2           45          83.284027         75.247970  persistence_raw
 3           60         111.344196         99.886624  persistence_raw}

In [17]:
# 1. On récupère le RMSE de la persistence CSI (ton champion actuel)
persistence_rmse = baseline_raw_results["persistence_csi"]["global"].query("metric == 'RMSE'")["value"].values[0]

# 2. On entraîne un modèle avec les nouvelles features
from sklearn.ensemble import RandomForestRegressor
model_adv = RandomForestRegressor(n_estimators=100, random_state=42)
model_adv.fit(X_train_adv, y_train_flat)

# 3. Évaluation
y_pred_adv = model_adv.predict(X_val_adv).reshape(val_target.shape)
res_adv = evaluate_model_bundle(val_target, y_pred_adv, model_name="RF_Advanced_Features")

# 4. Ajout des Skill Scores au tableau comparatif
import pandas as pd
all_results_df = pd.concat([bundle["global"] for bundle in baseline_raw_results.values()])
all_results_df = add_skill_scores(all_results_df, persistence_rmse)

display(all_results_df.sort_values("skill_score", ascending=False))

KeyError: 'persistence_csi'

## 4. Baseline 2 — persistance du clear-sky index

Deuxième baseline, plus cohérente avec la physique du problème :

- on suppose que le `CSI` reste constant à court terme ;
- on reconstruit ensuite `GHI` futur à partir du `CLS` futur.

$$
\widehat{CSI}_{t+h} = CSI_t
$$

puis

$$
\widehat{GHI}_{t+h} = \widehat{CSI}_{t+h} \times CLS_{t+h}
$$

In [ ]:
# %% Baseline persistance CSI

y_pred_persistence_csi = persistence_csi_baseline(val_arrays_raw)

print("Shape y_pred_persistence_csi :", y_pred_persistence_csi.shape)

baseline_csi_results = evaluate_model_bundle(
    val_target,
    y_pred_persistence_csi,
    model_name="persistence_csi",
)

print("Scores globaux")
display(baseline_csi_results["global"])

print("Scores par horizon")
display(baseline_csi_results["by_horizon"])

print("Scores sur moyenne spatiale")
display(baseline_csi_results["spatial_means"])

Shape y_pred_persistence_csi : (7, 4, 51, 51)
Scores globaux


,metric,value,model
0,RMSE,13.692846,persistence_csi
1,MAE,7.372206,persistence_csi


Scores par horizon


,horizon_min,RMSE,MAE,model
0,15,7.704670,4.140720,persistence_csi
1,30,11.665896,6.456973,persistence_csi
2,45,15.359845,8.693140,persistence_csi
3,60,17.849264,10.197991,persistence_csi


Scores sur moyenne spatiale


,horizon_min,RMSE_spatial_mean,MAE_spatial_mean,model
0,15,1.543409,1.294094,persistence_csi
1,30,2.888295,2.305033,persistence_csi
2,45,3.735279,2.803421,persistence_csi
3,60,4.933677,3.717848,persistence_csi


## 5. Baseline 3 — image moyenne du train

Cette baseline est volontairement très faible :
on prédit pour chaque exemple de validation la moyenne des cibles du train.

Elle sert surtout de borne basse de référence.

In [ ]:
# %% Baseline image moyenne

y_pred_mean = mean_image_baseline(train_target, n_samples=len(val_target))

baseline_mean_results = evaluate_model_bundle(
    val_target,
    y_pred_mean,
    model_name="mean_image",
)

print("Scores globaux")
display(baseline_mean_results["global"])

print("Scores par horizon")
display(baseline_mean_results["by_horizon"])

Scores globaux


,metric,value,model
0,RMSE,138.899038,mean_image
1,MAE,125.968348,mean_image


Scores par horizon


,horizon_min,RMSE,MAE,model
0,15,121.229709,117.251295,mean_image
1,30,130.792706,122.209406,mean_image
2,45,142.953652,126.661947,mean_image
3,60,157.900759,137.750746,mean_image


## 6. Comparaison initiale des baselines

On compare maintenant les trois baselines pour identifier la référence la plus difficile à battre.

In [ ]:
# %% Comparaison globale des baselines

baseline_results = {
    "persistence_raw": baseline_raw_results,
    "persistence_csi": baseline_csi_results,
    "mean_image": baseline_mean_results,
}

display(compare_global_scores(baseline_results))

,model,RMSE,MAE
0,persistence_csi,13.692846,7.372206
1,persistence_raw,77.652985,63.439056
2,mean_image,138.899038,125.968348


In [ ]:
# %% Comparaison par horizon

baseline_horizon_comparison = pd.concat(
    [
        baseline_raw_results["by_horizon"],
        baseline_csi_results["by_horizon"],
        baseline_mean_results["by_horizon"],
    ],
    ignore_index=True,
)

display(baseline_horizon_comparison)

,horizon_min,RMSE,MAE,model
0,15,28.998959,25.867481,persistence_raw
1,30,57.192327,51.134476,persistence_raw
2,45,84.976879,75.965643,persistence_raw
3,60,113.079497,100.788624,persistence_raw
4,15,7.704670,4.140720,persistence_csi
5,30,11.665896,6.456973,persistence_csi
6,45,15.359845,8.693140,persistence_csi
7,60,17.849264,10.197991,persistence_csi
8,15,121.229709,117.251295,mean_image
9,30,130.792706,122.209406,mean_image


## 7. Modèle 1 — Ridge Regression

Ridge est un premier modèle linéaire régularisé.

Il est particulièrement utile ici car :
- les features tabulaires sont nombreuses ;
- elles sont potentiellement corrélées ;
- la régularisation L2 améliore la stabilité de l'estimation.

In [ ]:
# %% Ridge

ridge_model = fit_ridge_multioutput(
    X_train_tab,
    y_train_flat,
    alpha=RIDGE_ALPHA,
    random_state=RANDOM_STATE,
)

y_pred_ridge = predict_tensor(ridge_model, X_val_tab)

ridge_results = evaluate_model_bundle(
    val_target,
    y_pred_ridge,
    model_name="ridge",
)

print("Scores globaux")
display(ridge_results["global"])

print("Scores par horizon")
display(ridge_results["by_horizon"])

Scores globaux


,metric,value,model
0,RMSE,31.092976,ridge
1,MAE,16.607825,ridge


Scores par horizon


,horizon_min,RMSE,MAE,model
0,15,30.383527,16.298581,ridge
1,30,30.990332,16.704750,ridge
2,45,31.285059,16.610637,ridge
3,60,31.698238,16.817330,ridge


In [ ]:
import gc
# Après avoir fini Ridge et stocké les scores
del ridge_model 
gc.collect()

## 8. Modèle 2 — Elastic Net

Elastic Net combine :
- la régularisation L1 (sélection partielle de variables),
- et la régularisation L2 (stabilité).

Il peut être utile lorsque les features sont nombreuses et partiellement redondantes.

In [ ]:
# %% Elastic Net

elasticnet_model = fit_elasticnet_multioutput(
    X_train_tab,
    y_train_flat,
    alpha=ELASTICNET_ALPHA,
    l1_ratio=ELASTICNET_L1_RATIO,
    random_state=RANDOM_STATE,
)

y_pred_elasticnet = predict_tensor(elasticnet_model, X_val_tab)

elasticnet_results = evaluate_model_bundle(
    val_target,
    y_pred_elasticnet,
    model_name="elasticnet",
)

print("Scores globaux")
display(elasticnet_results["global"])

print("Scores par horizon")
display(elasticnet_results["by_horizon"])

Scores globaux


,metric,value,model
0,RMSE,39.307231,elasticnet
1,MAE,22.815631,elasticnet


Scores par horizon


,horizon_min,RMSE,MAE,model
0,15,39.164402,22.886419,elasticnet
1,30,39.706616,23.412016,elasticnet
2,45,39.095447,22.681745,elasticnet
3,60,39.259573,22.282345,elasticnet


## 9. Modèle 3 — Random Forest

Random Forest est un premier modèle non linéaire capable de capturer :
- des interactions entre variables ;
- des effets non linéaires ;
- tout en restant relativement robuste.

In [ ]:
# %% Random Forest

rf_model = fit_random_forest_multioutput(
    X_train_tab,
    y_train_flat,
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=RANDOM_STATE,
)

y_pred_rf = predict_tensor(rf_model, X_val_tab)

rf_results = evaluate_model_bundle(
    val_target,
    y_pred_rf,
    model_name="random_forest",
)

print("Scores globaux")
display(rf_results["global"])

print("Scores par horizon")
display(rf_results["by_horizon"])

Scores globaux


,metric,value,model
0,RMSE,35.258683,random_forest
1,MAE,24.232948,random_forest


Scores par horizon


,horizon_min,RMSE,MAE,model
0,15,35.811201,24.973871,random_forest
1,30,35.566459,24.714165,random_forest
2,45,34.913329,23.903556,random_forest
3,60,34.732452,23.340200,random_forest


## 10. Modèle 4 — HistGradientBoosting

HistGradientBoosting est un modèle de boosting tabulaire souvent très performant,
notamment lorsqu'il existe des effets non linéaires complexes.

Dans cette première version, on l'utilise avec des hyperparamètres simples, sans tuning approfondi.

In [ ]:
# %% HistGradientBoosting

hgb_model = fit_hist_gb_multioutput(
    X_train_tab,
    y_train_flat,
    max_depth=HGB_MAX_DEPTH,
    learning_rate=HGB_LEARNING_RATE,
    max_iter=HGB_MAX_ITER,
    random_state=RANDOM_STATE,
)

y_pred_hgb = predict_tensor(hgb_model, X_val_tab)

hgb_results = evaluate_model_bundle(
    val_target,
    y_pred_hgb,
    model_name="hist_gradient_boosting",
)

print("Scores globaux")
display(hgb_results["global"])

print("Scores par horizon")
display(hgb_results["by_horizon"])

KeyboardInterrupt: 

## 11. Comparaison finale des modèles

On agrège maintenant les performances de toutes les baselines et de tous les modèles testés
pour comparer proprement les résultats.

In [ ]:
# %% Comparaison globale

all_results = {
    "persistence_raw": baseline_raw_results,
    "persistence_csi": baseline_csi_results,
    "mean_image": baseline_mean_results,
    "ridge": ridge_results,
    "elasticnet": elasticnet_results,
    "random_forest": rf_results,
    "hist_gradient_boosting": hgb_results,
}

global_comparison = compare_global_scores(all_results)
display(global_comparison)

NameError: name 'hgb_results' is not defined

In [ ]:
# %% Comparaison par horizon

horizon_comparison = pd.concat(
    [bundle["by_horizon"] for bundle in all_results.values()],
    ignore_index=True,
)

display(horizon_comparison)

In [ ]:
# %% Comparaison sur moyenne spatiale

spatial_mean_comparison = pd.concat(
    [bundle["spatial_means"] for bundle in all_results.values()],
    ignore_index=True,
)

display(spatial_mean_comparison)

## 12. Visualisation synthétique

On visualise ici les scores par horizon pour comparer la dégradation des performances
lorsque l'horizon de prévision augmente.

In [ ]:
# %% Courbe RMSE par horizon

fig, ax = plt.subplots(figsize=(8, 5))

for model_name, bundle in all_results.items():
    df = bundle["by_horizon"]
    ax.plot(df["horizon_min"], df["RMSE"], marker="o", label=model_name)

ax.set_xlabel("Horizon (minutes)")
ax.set_ylabel("RMSE")
ax.set_title("Comparaison des modèles par horizon - RMSE")
ax.legend()
plt.show()

In [ ]:
# %% Courbe MAE par horizon

fig, ax = plt.subplots(figsize=(8, 5))

for model_name, bundle in all_results.items():
    df = bundle["by_horizon"]
    ax.plot(df["horizon_min"], df["MAE"], marker="o", label=model_name)

ax.set_xlabel("Horizon (minutes)")
ax.set_ylabel("MAE")
ax.set_title("Comparaison des modèles par horizon - MAE")
ax.legend()
plt.show()

In [ ]:
# Comparaison visuelle pour le meilleur modèle
print("Analyse du modèle RF avec Features Avancées :")
plot_error_analysis(val_target, y_pred_adv, title="Random Forest : Réel vs Prédit")

## 13. Conclusion

Ce notebook a permis :

- de définir les métriques de comparaison ;
- d'établir plusieurs baselines de référence ;
- d'entraîner plusieurs modèles tabulaires supervisés ;
- de comparer leurs performances globales et par horizon.

### Points à retenir
- la baseline la plus forte donne la vraie référence à battre ;
- les modèles linéaires régularisés permettent de tester rapidement la valeur des features tabulaires ;
- les modèles non linéaires (Random Forest, boosting) permettent de capturer des interactions plus complexes.

### Suite logique
Le notebook suivant pourra approfondir :
- le tuning des hyperparamètres ;
- une validation temporelle plus robuste ;
- l'analyse des variables importantes ;
- et éventuellement une extension vers des modèles plus riches.